In [ ]:
import time
import cv2
import numpy as np
import keyboard
from sdlarch_rl.utils.utils import FrameSkip
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.stf6_imitation import STF6Env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
import pygame
from inputs import get_gamepad
from inputs import devices
import threading
from imitation.data import serialize
from imitation.data.types import Trajectory
import torch as th
import os
import re
import random

MAX_TRAJ=20

demo_path = 'demos/'

os.makedirs(demo_path, exist_ok=True)

last_index = -1

for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m:
        num = int(m.group(1))
        last_index = max(last_index, num)

print("last_index: " + str(last_index))

global myEnv

def make_env():
    def _init():
        global myEnv
        myEnv = STF6Env()

        env = WarpFrame(myEnv, width=64, height=64)
        # env = FrameSkip(env, skip=4)
        
        return env
    return _init

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4)

SCREEN_WIDTH = 640*3
SCREEN_HEIGHT = 480*3

pygame.init()

window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

prev_keys = set()

env.reset()

gamepad = devices.gamepads[0]

STATE = {
    "UP": 0, 
    "DOWN": 0, 
    "LEFT": 0, 
    "RIGHT": 0,
    "A": 0, 
    "B": 0, 
    "X": 0, 
    "START": 0,
}

lock = threading.Lock()

DEADZONE = 10000

def input_loop():
    while True:
        for e in gamepad.read():
            with lock:
                if e.code == "ABS_HAT0X":
                    STATE["LEFT"]  = 1 if e.state == -1 else 0
                    STATE["RIGHT"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_HAT0Y":
                    STATE["UP"]   = 1 if e.state == -1 else 0
                    STATE["DOWN"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_X":
                    if e.state > DEADZONE:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 1
                    elif e.state < -DEADZONE:
                        STATE["LEFT"] = 1
                        STATE["RIGHT"] = 0
                    else:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 0

                elif e.code == "ABS_Y":
                    if e.state > DEADZONE:
                        STATE["UP"] = 1
                        STATE["DOWN"] = 0
                    elif e.state < -DEADZONE:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 1
                    else:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 0

                elif e.code == "BTN_SOUTH":
                    STATE["A"] = e.state

                elif e.code == "BTN_EAST":
                    STATE["B"] = e.state

                elif e.code == "BTN_WEST":
                    STATE["X"] = e.state

                elif e.code == "BTN_START":
                    STATE["START"] = e.state


t = threading.Thread(target=input_loop, daemon=True)
t.start()

is_running = False
is_recording = False

color=(0, 0, 255)

count_record = 0

trajectories = []
# Data to store
recorded_obs = []
recorded_actions = []


while True:
    pygame.event.pump()
    
    action = [np.zeros(7, dtype=np.uint8)]

    if keyboard.is_pressed('k'):
        is_running = not is_running
        
        time.sleep(0.5)

        print("k was pressed ", is_running)

    if is_running:
        color=(0, 255, 0)
    else:
        color=(0, 0, 255)

    with lock:
        received_action = [
            STATE["UP"], 
            STATE["DOWN"], 
            STATE["LEFT"], 
            STATE["RIGHT"],
            STATE["A"], 
            STATE["B"], 
            STATE["X"]
        ]

        if keyboard.is_pressed('up') or received_action[0]: action[0][0] = 1
        if keyboard.is_pressed('down') or received_action[1]: action[0][1] = 1
        if keyboard.is_pressed('left') or received_action[2]: action[0][2] = 1
        if keyboard.is_pressed('right') or received_action[3]: action[0][3] = 1
        if keyboard.is_pressed('i') or received_action[4]: action[0][4] = 1
        if keyboard.is_pressed('o') or received_action[5]: action[0][5] = 1
        if keyboard.is_pressed('p') or received_action[6]: action[0][6] = 1

    if is_running != is_recording:
        if is_running:
            print("first obs")
            recorded_obs.append(obs)

    obs, _, _, _ = env.step(action)

    img = myEnv.render()

    if img is not None:
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        text_to_display = "Count: " + str(count_record)
        position = (50, 250)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        thickness = 2
        line_type = cv2.LINE_AA


        cv2.putText(img, text_to_display, position, font, font_scale, color, thickness, line_type)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))

        window.blit(surface, (0, 0))
        
    pygame.display.update()

    received_action = np.random.choice([0, 1], size=7)

    if is_running != is_recording:
        if not is_running:
            print("end recording, increase count")
            count_record = count_record + 1

            # print(recorded_obs[0])

            obs_uint8 = np.stack(
                [obs.astype(np.uint8) for obs in recorded_obs],
                axis=0
            )
            print(obs_uint8.shape)

            trajectories.append(
                Trajectory(
                    obs=obs_uint8,
                    acts=np.array(recorded_actions), 
                    infos = None, 
                    terminal = False
                )
            )

            recorded_obs = []
            recorded_actions = []

    is_recording = is_running

    if is_running:
        recorded_obs.append(obs)
        recorded_actions.append(action)

    if count_record > MAX_TRAJ - 1:
        print("end recording")
        break

print(len(trajectories))

th.save(trajectories, demo_path + f"demos{last_index + 1}.pt")

pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 28
next search...
next search...
next search...
next search...
next search...
next search...
Window founded HWND: 2625530.
reset obs shape (128, 228, 3)
